**Theme:**

*“Training loss means nothing without validation.”*

In [1]:
import torch 
import torch.nn as nn 

torch.__version__

'2.8.0+cu129'

# Validation, Overfitting, Model saving

## Add validation data

In [12]:
# Setting manual seed for regeneration purpose
torch.manual_seed(11) 

# Data
x_train = torch.tensor([1., 2., 3., 4.])
y_train = torch.tensor([3., 5., 7., 9.])

x_val = torch.tensor([5., 6.])
y_val = torch.tensor([11., 13.])

# Model Definition 
class LinearModel(nn.Module): 

    def __init__(self):
        super().__init__() 
        self.w = nn.Parameter(torch.randn(1)) 
        self.b = nn.Parameter(torch.randn(1)) 
    def forward(self,x): 
        return self.w * x + self.b 
    

model = LinearModel() 
loss_fn = nn.MSELoss() 
optimizer = torch.optim.SGD(model.parameters(), lr=0.1) 

print(f"w: {model.w.item():.4f}, b: {model.b.item():.4f}")

w: 0.7376, b: 1.9459


## Training with validation

In [13]:
epochs = 50 

for epoch in range(epochs): 

    # TRAIN 
    y_pred = model(x_train) 
    loss = loss_fn(y_train, y_pred) 

    optimizer.zero_grad()
    loss.backward() 
    optimizer.step() 

    # VALIDATION   
    with torch.no_grad():
        val_pred = model(x_val) 
        val_loss = loss_fn(y_val, val_pred) 

    # Printing the loss in every 10 epochs
    if not (epoch+1)%10:
        print(f"Epoch: {epoch+1}, Training Loss: {loss.item():.3f}, Validation Loss: {val_loss.item():.3f}")

Epoch: 10, Training Loss: 0.152, Validation Loss: 0.804
Epoch: 20, Training Loss: 0.080, Validation Loss: 0.356
Epoch: 30, Training Loss: 0.044, Validation Loss: 0.193
Epoch: 40, Training Loss: 0.024, Validation Loss: 0.105
Epoch: 50, Training Loss: 0.013, Validation Loss: 0.057


- Which loss is more important?

- Why do we use `torch.no_grad()`?

## Simulate overfitting

In [18]:
model2 = LinearModel() 
optimizer2 = torch.optim.SGD(model2.parameters(), lr=0.1)

epochs = 200 

for epoch in range(epochs): 

    # TRAIN 
    y_pred = model2(x_train) 
    loss = loss_fn(y_train, y_pred) 

    optimizer2.zero_grad()
    loss.backward() 
    optimizer2.step() 

    # VALIDATION   
    with torch.no_grad():
        val_pred = model2(x_val) 
        val_loss = loss_fn(y_val, val_pred) 

    # Printing the loss in every 10 epochs
    if not (epoch+1)%10:
        print(f"Epoch: {epoch+1}, Training Loss: {loss.item():.6f}, Validation Loss: {val_loss.item():.6f}")

Epoch: 10, Training Loss: 0.106549, Validation Loss: 0.337709
Epoch: 20, Training Loss: 0.055314, Validation Loss: 0.243639
Epoch: 30, Training Loss: 0.030116, Validation Loss: 0.133519
Epoch: 40, Training Loss: 0.016398, Validation Loss: 0.072711
Epoch: 50, Training Loss: 0.008928, Validation Loss: 0.039590
Epoch: 60, Training Loss: 0.004861, Validation Loss: 0.021556
Epoch: 70, Training Loss: 0.002647, Validation Loss: 0.011736
Epoch: 80, Training Loss: 0.001441, Validation Loss: 0.006390
Epoch: 90, Training Loss: 0.000785, Validation Loss: 0.003479
Epoch: 100, Training Loss: 0.000427, Validation Loss: 0.001894
Epoch: 110, Training Loss: 0.000233, Validation Loss: 0.001031
Epoch: 120, Training Loss: 0.000127, Validation Loss: 0.000562
Epoch: 130, Training Loss: 0.000069, Validation Loss: 0.000306
Epoch: 140, Training Loss: 0.000038, Validation Loss: 0.000167
Epoch: 150, Training Loss: 0.000020, Validation Loss: 0.000091
Epoch: 160, Training Loss: 0.000011, Validation Loss: 0.000049
E

- What happens to train loss?

- What happens to validation loss?

## Save the model

In [19]:
torch.save(model2.state_dict(), 'linear_model.pth') 


In [22]:
new_model = LinearModel() 

new_model.load_state_dict(torch.load('linear_model.pth')) 

pred_new = new_model(torch.tensor(10.0)) 
pred_2 = model2(torch.tensor(10.0))  

print(f"Predictions -> Before model saved: {pred_2.item():.5f}, After model saved and load: {pred_new.item():.5f}")

Predictions -> Before model saved: 21.00564, After model saved and load: 21.00564


- Why don’t we save the whole model?

- What is `state_dict`?

---

Why is validation loss more important than training loss?

# Interaction

### 1️⃣ Why training loss can increase after going down



Even after convergence, we still use gradient descent, which is:

$𝑤 = 𝑤 − \eta⋅∇𝑤$


This is a step, not a jump to the minimum.

If:

- Learning rate is too high

- Or gradients are noisy (mini-batches)

then the optimizer can overshoot the minimum.

So instead of going:
```sh
↓ ↓ ↓ ↓
```
it does:
```bash
↓ ↓ ↑ ↓ ↑
```
This causes training loss to go up and down even near convergence.

This is completely normal.
***

### 2️⃣ Why validation loss increases while training loss decreases



#### This is overfitting.

What happens:

- Model memorizes training data

- Stops learning general patterns

- Starts fitting noise

So:

- Training loss → ↓

-Validation loss → ↑

This is why:

- The best model is NOT the one at the last epoch, but the one with the lowest validation loss.

This is why we use:

- Early stopping

- Checkpointing
***

### 3️⃣ Why don't we save the whole model?

We usually save:

- Architecture in code

- Weights in `state_dict`

This allows:

- Rebuilding model

- Loading trained weights

This is how production systems work.

***